# Chapter 3: Recompute the region evaluator

**The question.** Chapter 3 defines how a region is scored: who belongs to it, the model's error (Brier loss) inside and outside it, and the difference between the two (the contrast). This notebook checks that the saved scores of the fresh run follow directly from its raw data.

**What was run live.** Run 20260923T220909Z repeated the whole study on two machines. `spark-fe12` ran the numerical search with three new seeds (74101 to 74103), one live structural proposal from a language model and 200,000 new confirmation customers. `spark-ef84` ran 70 live answer tests and nine report drafts with `claude-sonnet-5`. The simulated population and the frozen money-laundering model are the same as in the `regional_case` notebooks.

**Result.** All 13 saved region scores are reproduced from the 200,000 new customers.

The seeds, the proposal and the confirmation sample are new, so do not mix these numbers with those in the `regional_case` notebooks. This notebook reads recorded results; rerunning it makes no language-model call.

## Setup and source checks

Finds the saved results of this run and checks their SHA-256 fingerprint, so every value below comes from the unchanged run.

In [1]:
from pathlib import Path
import json, hashlib, os, platform
import numpy as np
import pandas as pd

# Locate the data root: the directory whose relative layout the evidence records use.
MARKER = "book/notebooks/regional_case/companion.py"

def find_data_root():
    configured = os.environ.get("MODEL_VALIDATION_ROOT")
    if configured:
        root = Path(configured).expanduser().resolve()
        if not (root / MARKER).is_file():
            raise FileNotFoundError(f"MODEL_VALIDATION_ROOT must contain {MARKER}")
        return root, "MODEL_VALIDATION_ROOT"
    for path in (Path.cwd(), *Path.cwd().parents):
        if (path / MARKER).is_file():
            return path, "repository checkout"
    try:
        from aiv.companions import locate
    except ImportError:
        raise FileNotFoundError(
            "Companion data not found. Install the aiv package, open the notebook "
            "inside the repository checkout, or set MODEL_VALIDATION_ROOT.") from None
    return locate()

ROOT, DATA_SOURCE = find_data_root()
LIVE = ROOT / "book/experiments/regional_live"
R = ROOT / "book/experiments/regional_live/runs/20260923T220909Z"
print("Python:", platform.python_version())
print("Data root found from:", DATA_SOURCE)
print("Run:", R.relative_to(ROOT).as_posix())
read=lambda p: json.loads((R/p).read_text(encoding="utf-8"))
results=read('regional/results.json')
reference=read('reference.json')
seeds=reference['seeds']
assert hashlib.sha256((R/'regional/results.json').read_bytes()).hexdigest()==reference['source_sha256']


Python: 3.12.3
Data root found from: repository checkout
Run: book/experiments/regional_live/runs/20260923T220909Z


## Recompute every region score

For each of the 200,000 new customers the cell computes the error $L_i=(y_i-p_i)^2$ from the outcome $y_i$ (0 or 1) and the model's probability $p_i$. For each frozen region it then recomputes three numbers from the region's membership list and checks them against the saved results: its share of customers, its average error and its contrast (average error inside minus outside).

In [2]:
a=np.load(R/'regional/confirmation.npz')
loss=(a['y']-a['p'])**2
for key,row in results['confirmation'].items():
 mask=a[key]
 assert np.isclose(mask.mean(),row['share'])
 assert np.isclose(loss[mask].mean(),row['brier'])
 assert np.isclose(loss[mask].mean()-loss[~mask].mean(),row['contrast'])
print('All',len(results['confirmation']),'regions verified on',len(loss),'observations')

All 13 regions verified on 200000 observations


**Read the output.** **In one line: all 13 saved region scores follow directly from the raw data.**

**Notice.** Region membership was fixed from customer features before these outcomes were drawn, so the outcomes could not have influenced which regions were chosen.